# Module 9 · Solutions

In [ ]:
import pandas as pd, numpy as np
import warnings; warnings.filterwarnings("ignore")
rng = np.random.default_rng(9)
months = pd.date_range("2020-01-31", periods=72, freq="ME")
true_trend = 520 + 4.2*np.arange(72)
sidx = {1:0.94,2:0.92,3:0.98,4:1.00,5:0.97,6:0.95,7:0.98,8:1.02,9:1.05,10:1.22,11:1.14,12:0.83}
true_seasonal = np.array([sidx[m.month] for m in months])
sales = pd.Series(true_trend*true_seasonal*rng.normal(1.0,0.03,72), index=months)

## 9A

In [ ]:
# Ex1 - additive world, multiplicative tool
rng2 = np.random.default_rng(9)
centered = np.array([sidx[m]-1 for m in range(1,13)])
sales_add = pd.Series(true_trend + 60*np.tile(centered*4, 6)[:72] + rng2.normal(0,15,72), index=months)
trend_est = sales_add.rolling(12, center=True).mean()
est = (sales_add/trend_est).groupby(lambda d:d.month).mean()
print("Multiplicative index recovered from an ADDITIVE world (early vs late years differ!):")
print((est/est.mean()).round(3).to_string())
print("A fixed +Rs lift is a LARGE % of a small early base and a SMALL % of the big late base ->")
print("the ratio-based index is a blend that fits no single year. Wrong-model error is systematic, not noisy.")

# Ex2 - the December defence
nov, dec = 905, 640
print(f"\nDeseasonalised Nov: {nov/sidx[11]:,.0f} | Dec: {dec/sidx[12]:,.0f}")
print("Trend-level Dec is ABOVE trend-level Nov - the business strengthened while raw sales 'fell 30%'.")

# Ex3 - level shift
shifted = sales.copy(); shifted.iloc[40:] += 120
t2 = shifted.rolling(12, center=True).mean()
resid = shifted/(t2*pd.Series([sidx[m.month] for m in months], index=months))
print(f"\nResidual around the break (months 36-46):")
print(resid.iloc[36:46].round(3).to_string())
print("The break smears into the TREND (the MA ramps for 12 months instead of stepping) and leaves a")
print("residual wave around month 40. Lesson: eyeball for breaks BEFORE decomposing - the tool assumes none.")

## 9B

In [ ]:
# Ex1 - sales ACF
lags = range(1,25)
acf = [sales.autocorr(k) for k in lags]
tall = [k for k,v in zip(lags,acf) if v > 0.5]
print("Sales ACF tall bars at lags:", tall, "- 12 and 24 dominate: the season IS the memory.")

# Ex2 - differenced sales
d = sales.pct_change().dropna()
print(f"\nDiff'd sales rolling-mean range: {d.rolling(12).mean().min():.3f} to {d.rolling(12).mean().max():.3f} (hugs ~0.006: stationary level)")
print("ACF of diff'd sales at lag 12:", round(d.autocorr(12),3), "- the SEASON survives differencing.")
print("That's not a bug: differencing kills trend, not season. The surviving lag-12 spike is exactly what")
print("SARIMA's seasonal terms exist to model - it's the signal.")

# Ex3 - calibrating the eye
for seed in range(5):
    noise = pd.Series(np.random.default_rng(seed).normal(0,1,1200))
    band = 2/np.sqrt(1200)
    esc = sum(abs(noise.autocorr(k)) > band for k in range(1,25))
    print(f"seed {seed}: {esc}/24 bars escape the band (chance predicts ~1)")

## 9C

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing
# Ex1 - walk-forward HW vs seasonal naive
errs_hw, errs_sn = [], []
for i in range(54, 72):
    tr, actual = sales.iloc[:i], sales.iloc[i]
    hw = ExponentialSmoothing(tr, trend="add", seasonal="mul", seasonal_periods=12).fit()
    errs_hw.append(abs(float(hw.forecast(1).iloc[0]) - actual))
    errs_sn.append(abs(tr.iloc[-12] - actual))
print(f"Walk-forward MAE (18 steps): HW {np.mean(errs_hw):.1f} vs seasonal naive {np.mean(errs_sn):.1f}")
print("HW holds its edge in walk-forward - the single split wasn't flattering it. Structure is real here.")

# Ex2 - level shift resilience
sh = sales.copy(); sh.iloc[40:] += 120
errs = {}
for name, a in [("alpha via fit", None)]:
    pass
e_hw, e_sn = [], []
for i in range(54, 72):
    tr, actual = sh.iloc[:i], sh.iloc[i]
    m_ = ExponentialSmoothing(tr, trend="add", seasonal="mul", seasonal_periods=12).fit()
    e_hw.append(abs(float(m_.forecast(1).iloc[0]) - actual))
    e_sn.append(abs(tr.iloc[-12] - actual))
print(f"\nWith a break at month 40: HW MAE {np.mean(e_hw):.1f} vs seasonal naive {np.mean(e_sn):.1f}")
print("Smoothing adapts within months (alpha = a forgetting rate); seasonal naive stays wrong for a full")
print("year (it quotes the pre-break same-month). Adaptivity is smoothing's quiet superpower after breaks.")

In [ ]:
# Ex3 - the un-forecastable, confirmed
from statsmodels.tsa.statespace.sarimax import SARIMAX
BASE = "data/"
px = pd.read_csv(BASE+"nifty50_prices.csv", parse_dates=["date"]).set_index("date")
mm = px["close"].resample("ME").last().dropna()
tr, te = mm.iloc[:48], mm.iloc[48:]
sar = SARIMAX(tr, order=(1,1,1), seasonal_order=(1,1,1,12)).fit(disp=False)
f = sar.forecast(len(te)); nv = pd.Series(tr.iloc[-1], index=te.index)
print(f"NIFTY monthly - SARIMA RMSE {np.sqrt(((te-f)**2).mean()):,.0f} vs naive {np.sqrt(((te-nv)**2).mean()):,.0f}")
print("\nThe two-sentence conclusion: the identical pipeline that beat the floor on sales cannot beat")
print("'tomorrow = today' on an index. Forecastability is a property of the SERIES, not of the model.")